In [ ]:
# 1. Configurar entorno Azure ML
import tensorflow as tf
import time
import os
import subprocess

print(f"TensorFlow: {tf.__version__}")
print(f"Num GPUs: {len(tf.config.list_physical_devices('GPU'))}")

# Detectar GPU
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    # Obtener info de GPU
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], 
                          capture_output=True, text=True)
    gpu_info = result.stdout.strip()
    print(f"GPU detectada: {gpu_info}")
    
    # Habilitar memory growth
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    
    # Detectar tipo de GPU Azure
    gpu_name = gpu_info.split(',')[0] if gpu_info else ""
    
    if "A100" in gpu_name:
        print("🔥 A100 detectada (NC24ads_A100_v4) - ~15-20 min")
        BATCH_SIZE = 128  # A100 tiene 80GB!
        MIXED_PRECISION = True
    elif "V100" in gpu_name:
        print("⚡ V100 detectada (NC6s_v3/NC12s_v3) - ~40-50 min")
        BATCH_SIZE = 64
        MIXED_PRECISION = True
    elif "T4" in gpu_name:
        print("🔧 T4 detectada - ~60-90 min")
        BATCH_SIZE = 32
        MIXED_PRECISION = True
    elif "K80" in gpu_name:
        print("⚠️ K80 detectada (NC6) - ~2-3 horas (considera upgrade)")
        BATCH_SIZE = 16
        MIXED_PRECISION = False
    else:
        print(f"🔧 GPU: {gpu_name}")
        BATCH_SIZE = 32
        MIXED_PRECISION = True
else:
    print("❌ SIN GPU - Asegúrate de usar un compute con GPU")
    print("   Opciones recomendadas:")
    print("   - Standard_NC6s_v3 (V100, 16GB)")
    print("   - Standard_NC24ads_A100_v4 (A100, 80GB)")
    BATCH_SIZE = 16
    MIXED_PRECISION = False

# Habilitar mixed precision para GPUs modernas
if MIXED_PRECISION:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print("✅ Mixed precision habilitado (float16)")

print(f"\nBatch size configurado: {BATCH_SIZE}")

In [ ]:
# 2. Instalar dependencias
!pip install -q tensorflow-datasets gdown albumentations

import numpy as np
import json
import shutil
from pathlib import Path

# Configuración optimizada para Azure (más epochs con GPU potente)
CONFIG = {
    'IMG_SIZE': 224,
    'EPOCHS_PHASE1': 25,      # Más epochs en Azure
    'EPOCHS_PHASE2': 35,      # Fine-tuning más largo
    'LEARNING_RATE_1': 1e-3,
    'LEARNING_RATE_2': 1e-5,
    'LABEL_SMOOTHING': 0.1,
    'DROPOUT_RATE': 0.4,
    'L2_REG': 0.01,
}

# Directorio de trabajo
UNIFIED_DIR = Path('colombia_crops_dataset')
UNIFIED_DIR.mkdir(exist_ok=True)

# En Azure, usar directorio de outputs para persistencia
OUTPUT_DIR = Path('./outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

print("✅ Configuración Azure ML:")
for k, v in CONFIG.items():
    print(f"   {k}: {v}")

In [ ]:
# 3. Descargar PlantVillage (cultivos base)
import urllib.request
import zipfile

print("="*60)
print("🥔🌽🍅 DESCARGANDO PLANTVILLAGE")
print("="*60)

PLANTVILLAGE_URL = "https://github.com/spMohanty/PlantVillage-Dataset/archive/refs/heads/master.zip"

if not os.path.exists("PlantVillage-Dataset-master"):
    print("📥 Descargando PlantVillage (~250MB)...")
    start = time.time()
    urllib.request.urlretrieve(PLANTVILLAGE_URL, "plantvillage.zip")
    print(f"   Descargado en {time.time()-start:.1f}s")
    
    print("📦 Extrayendo...")
    with zipfile.ZipFile("plantvillage.zip", 'r') as z:
        z.extractall(".")
    os.remove("plantvillage.zip")
else:
    print("✅ PlantVillage ya descargado")

# Cultivos colombianos de PlantVillage
COLOMBIAN_CROPS = ['Potato', 'Corn', 'Tomato', 'Pepper', 'Orange']

CROP_TRANSLATIONS = {
    'Potato': 'Papa',
    'Corn': 'Maiz',
    'Tomato': 'Tomate',
    'Pepper': 'Pimiento',
    'Orange': 'Naranja',
}

DISEASE_TRANSLATIONS = {
    'healthy': 'Saludable',
    'Early_blight': 'Tizon_temprano',
    'Late_blight': 'Tizon_tardio',
    'Common_rust_': 'Roya_comun',
    'Northern_Leaf_Blight': 'Tizon_norteno',
    'Cercospora_leaf_spot Gray_leaf_spot': 'Mancha_gris',
    'Bacterial_spot': 'Mancha_bacteriana',
    'Septoria_leaf_spot': 'Septoriosis',
    'Leaf_Mold': 'Moho_foliar',
    'Target_Spot': 'Mancha_diana',
    'Spider_mites Two-spotted_spider_mite': 'Acaros',
    'Tomato_Yellow_Leaf_Curl_Virus': 'Virus_rizado_amarillo',
    'Tomato_mosaic_virus': 'Virus_mosaico',
    'Haunglongbing_(Citrus_greening)': 'Huanglongbing',
}

PV_DIR = Path("PlantVillage-Dataset-master/raw/color")
for folder in sorted(os.listdir(PV_DIR)):
    for crop in COLOMBIAN_CROPS:
        if folder.startswith(crop):
            parts = folder.split('___')
            crop_name = CROP_TRANSLATIONS.get(parts[0], parts[0])
            disease = parts[1] if len(parts) > 1 else 'healthy'
            disease_name = DISEASE_TRANSLATIONS.get(disease, disease.replace('_', ' '))
            
            new_name = f"{crop_name}___{disease_name}"
            src = PV_DIR / folder
            dst = UNIFIED_DIR / new_name
            
            if not dst.exists() and src.exists():
                shutil.copytree(src, dst)
                count = len(list(dst.glob('*')))
                print(f"   ✅ {new_name}: {count} imgs")

print("\n✅ PlantVillage procesado")

In [ ]:
# 4. Descargar dataset de Yuca/Cassava
print("="*60)
print("🥬 DESCARGANDO DATASET DE YUCA (CASSAVA)")
print("="*60)

import tensorflow_datasets as tfds

cassava_classes = {
    0: 'Yuca___Anublo_bacterial',
    1: 'Yuca___Rayado_marron', 
    2: 'Yuca___Acaro_verde',
    3: 'Yuca___Mosaico',
    4: 'Yuca___Saludable',
}

# Crear directorios
for class_name in cassava_classes.values():
    (UNIFIED_DIR / class_name).mkdir(exist_ok=True)

# Verificar si ya existe
yuca_count = sum(len(list((UNIFIED_DIR / c).glob('*'))) for c in cassava_classes.values())
if yuca_count < 1000:
    print("📥 Descargando Cassava desde TensorFlow Datasets...")
    cassava_ds, cassava_info = tfds.load('cassava', with_info=True, as_supervised=True)
    
    print("📁 Guardando imágenes...")
    for split in ['train', 'validation', 'test']:
        if split in cassava_ds:
            for i, (image, label) in enumerate(cassava_ds[split]):
                class_name = cassava_classes[int(label)]
                img_path = UNIFIED_DIR / class_name / f"{split}_{i}.jpg"
                tf.io.write_file(str(img_path), tf.io.encode_jpeg(image))
                if i % 1000 == 0:
                    print(f"   Procesadas {i} imágenes de {split}...")
else:
    print("✅ Yuca ya descargada")

# Mostrar conteo
for class_name in cassava_classes.values():
    count = len(list((UNIFIED_DIR / class_name).glob('*')))
    print(f"   ✅ {class_name}: {count} imgs")

print("\n✅ Yuca/Cassava procesado")

In [ ]:
# 5. Descargar dataset de Café (BRACOL/LARA)
print("="*60)
print("☕ DESCARGANDO DATASET DE CAFÉ")
print("="*60)

coffee_mapping = {
    'healthy': 'Cafe___Saludable',
    'rust': 'Cafe___Roya',
    'miner': 'Cafe___Minador',
    'cercospora': 'Cafe___Cercospora',
    'phoma': 'Cafe___Phoma',
    'red_spider_mite': 'Cafe___Arana_roja'
}

# Verificar si ya existe
cafe_exists = any((UNIFIED_DIR / c).exists() for c in coffee_mapping.values())

if not cafe_exists:
    print("📥 Descargando dataset BRACOL de café...")
    !git clone --depth 1 https://github.com/esgario/lara2018.git coffee_temp 2>/dev/null || echo "Clonando..."
    
    if os.path.exists('coffee_temp/dataset'):
        for folder in os.listdir('coffee_temp/dataset'):
            src = Path('coffee_temp/dataset') / folder
            if src.is_dir():
                folder_lower = folder.lower().replace(' ', '_')
                new_name = coffee_mapping.get(folder_lower)
                if new_name:
                    dst = UNIFIED_DIR / new_name
                    if not dst.exists():
                        shutil.copytree(src, dst)
                        count = len(list(dst.glob('*')))
                        print(f"   ✅ {new_name}: {count} imgs")
        !rm -rf coffee_temp
    else:
        print("⚠️ Estructura de café no encontrada, usando dataset alternativo...")
        # Fallback: crear clases vacías que se llenarán con augmentation
        for name in coffee_mapping.values():
            (UNIFIED_DIR / name).mkdir(exist_ok=True)
else:
    print("✅ Café ya descargado")

print("\n✅ Café procesado")

In [ ]:
# 6. Verificar dataset unificado
print("="*60)
print("📊 RESUMEN DEL DATASET COLOMBIANO")
print("="*60)

all_classes = sorted([d.name for d in UNIFIED_DIR.iterdir() if d.is_dir()])

# Filtrar clases vacías
valid_classes = []
for class_name in all_classes:
    count = len([f for f in (UNIFIED_DIR / class_name).iterdir() if f.is_file()])
    if count >= 10:  # Mínimo 10 imágenes
        valid_classes.append(class_name)

print(f"\n📁 {len(valid_classes)} clases válidas (≥10 imágenes):\n")

total_images = 0
crops_summary = {}

for class_name in valid_classes:
    count = len([f for f in (UNIFIED_DIR / class_name).iterdir() if f.is_file()])
    total_images += count
    
    crop = class_name.split('___')[0]
    if crop not in crops_summary:
        crops_summary[crop] = {'classes': 0, 'images': 0}
    crops_summary[crop]['classes'] += 1
    crops_summary[crop]['images'] += count
    
    print(f"  {class_name}: {count:,} imágenes")

print(f"\n{'='*60}")
print(f"📊 RESUMEN POR CULTIVO:\n")
for crop, data in sorted(crops_summary.items()):
    print(f"  🌱 {crop}: {data['classes']} clases, {data['images']:,} imágenes")

print(f"\n{'='*60}")
print(f"📷 TOTAL: {total_images:,} imágenes en {len(valid_classes)} clases")

NUM_CLASSES = len(valid_classes)

In [ ]:
# 7. Crear generadores de datos optimizados para Azure
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = CONFIG['IMG_SIZE']

# Augmentation agresiva
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.6, 1.4],
    channel_shift_range=30,
    fill_mode='reflect',
    validation_split=0.15
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.15
)

print(f"🔄 Creando generadores con batch_size={BATCH_SIZE}...")

train_gen = train_datagen.flow_from_directory(
    str(UNIFIED_DIR),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

val_gen = val_datagen.flow_from_directory(
    str(UNIFIED_DIR),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

NUM_CLASSES = len(train_gen.class_indices)
STEPS_PER_EPOCH = train_gen.samples // BATCH_SIZE
VAL_STEPS = val_gen.samples // BATCH_SIZE

print(f"\n📊 Dataset split:")
print(f"   Training: {train_gen.samples:,} imágenes ({STEPS_PER_EPOCH} steps/epoch)")
print(f"   Validation: {val_gen.samples:,} imágenes ({VAL_STEPS} steps)")
print(f"   Clases: {NUM_CLASSES}")

In [ ]:
# 8. Crear modelo MobileNetV2 optimizado
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2

print("🏗️ Creando modelo MobileNetV2...")

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False

# Cabeza de clasificación robusta
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(1024, activation='relu', kernel_regularizer=l2(CONFIG['L2_REG']))(x)
x = Dropout(CONFIG['DROPOUT_RATE'])(x)
x = BatchNormalization()(x)
x = Dense(512, activation='relu', kernel_regularizer=l2(CONFIG['L2_REG']))(x)
x = Dropout(CONFIG['DROPOUT_RATE'])(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu', kernel_regularizer=l2(CONFIG['L2_REG']))(x)
x = Dropout(0.3)(x)

# Output layer - usar float32 para estabilidad con mixed precision
predictions = Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

model = Model(inputs=base_model.input, outputs=predictions)

print(f"\n📋 Arquitectura:")
print(f"   Parámetros totales: {model.count_params():,}")
print(f"   Clases de salida: {NUM_CLASSES}")

model.summary(show_trainable=True, expand_nested=False)

In [ ]:
# 9. FASE 1: Entrenar clasificador
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard
from tensorflow.keras.losses import CategoricalCrossentropy

loss_fn = CategoricalCrossentropy(label_smoothing=CONFIG['LABEL_SMOOTHING'])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG['LEARNING_RATE_1']),
    loss=loss_fn,
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

# Callbacks con TensorBoard para Azure ML
log_dir = OUTPUT_DIR / 'logs' / f"phase1_{time.strftime('%Y%m%d-%H%M%S')}"
callbacks_phase1 = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    ModelCheckpoint(str(OUTPUT_DIR / 'best_phase1.keras'), monitor='val_accuracy', save_best_only=True, verbose=1),
    TensorBoard(log_dir=str(log_dir), histogram_freq=1)
]

print("="*60)
print("🚀 FASE 1: Entrenando clasificador (base congelada)")
print("="*60)
print(f"   Epochs: {CONFIG['EPOCHS_PHASE1']}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Learning rate: {CONFIG['LEARNING_RATE_1']}")
print()

start_time = time.time()

history1 = model.fit(
    train_gen,
    epochs=CONFIG['EPOCHS_PHASE1'],
    validation_data=val_gen,
    callbacks=callbacks_phase1,
    verbose=1
)

phase1_time = time.time() - start_time
phase1_acc = max(history1.history['val_accuracy'])

print(f"\n✅ Fase 1 completada en {phase1_time/60:.1f} minutos")
print(f"   Best Val Accuracy: {phase1_acc:.2%}")

In [ ]:
# 10. FASE 2: Fine-tuning
print("="*60)
print("🔧 FASE 2: Fine-tuning")
print("="*60)

model.load_weights(str(OUTPUT_DIR / 'best_phase1.keras'))

# Descongelar últimas capas
base_model.trainable = True
FREEZE_UNTIL = 100

for layer in base_model.layers[:FREEZE_UNTIL]:
    layer.trainable = False

trainable_params = sum(tf.keras.backend.count_params(w) for w in model.trainable_weights)
print(f"   Capas descongeladas: {len(base_model.layers) - FREEZE_UNTIL}")
print(f"   Parámetros entrenables: {trainable_params:,}")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG['LEARNING_RATE_2']),
    loss=loss_fn,
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

log_dir = OUTPUT_DIR / 'logs' / f"phase2_{time.strftime('%Y%m%d-%H%M%S')}"
callbacks_phase2 = [
    EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint(str(OUTPUT_DIR / 'best_phase2.keras'), monitor='val_accuracy', save_best_only=True, verbose=1),
    TensorBoard(log_dir=str(log_dir), histogram_freq=1)
]

print(f"\n   Epochs: {CONFIG['EPOCHS_PHASE2']}")
print(f"   Learning rate: {CONFIG['LEARNING_RATE_2']}")
print()

start_time = time.time()

history2 = model.fit(
    train_gen,
    epochs=CONFIG['EPOCHS_PHASE2'],
    validation_data=val_gen,
    callbacks=callbacks_phase2,
    verbose=1
)

phase2_time = time.time() - start_time
final_acc = max(history2.history['val_accuracy'])
final_top3 = max(history2.history['val_top3_acc'])
total_time = phase1_time + phase2_time

print(f"\n✅ Fase 2 completada en {phase2_time/60:.1f} minutos")
print(f"   Best Val Accuracy: {final_acc:.2%}")
print(f"   Best Val Top-3 Accuracy: {final_top3:.2%}")
print(f"\n⏱️ Tiempo total: {total_time/60:.1f} minutos")

In [ ]:
# 11. Evaluación y métricas
import matplotlib.pyplot as plt

model.load_weights(str(OUTPUT_DIR / 'best_phase2.keras'))

print("="*60)
print("📊 EVALUACIÓN FINAL")
print("="*60)

val_gen.reset()
results = model.evaluate(val_gen, verbose=0)
print(f"\n🎯 Métricas:")
print(f"   Loss: {results[0]:.4f}")
print(f"   Accuracy: {results[1]:.2%}")
print(f"   Top-3 Accuracy: {results[2]:.2%}")

# Análisis de confianza
val_gen.reset()
y_pred_probs = model.predict(val_gen, verbose=1)
confidence_scores = np.max(y_pred_probs, axis=1)

print(f"\n📈 Distribución de confianza:")
print(f"   Media: {np.mean(confidence_scores):.2%}")
print(f"   >70%: {np.mean(confidence_scores > 0.7):.1%}")
print(f"   >90%: {np.mean(confidence_scores > 0.9):.1%}")

# Gráficas
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

offset = len(history1.history['accuracy'])
axes[0].plot(history1.history['accuracy'], label='Train P1')
axes[0].plot(history1.history['val_accuracy'], label='Val P1')
axes[0].plot(range(offset, offset+len(history2.history['accuracy'])), history2.history['accuracy'], label='Train P2')
axes[0].plot(range(offset, offset+len(history2.history['val_accuracy'])), history2.history['val_accuracy'], label='Val P2')
axes[0].axvline(x=offset-0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].set_title('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history1.history['loss'], label='Train P1')
axes[1].plot(history1.history['val_loss'], label='Val P1')
axes[1].plot(range(offset, offset+len(history2.history['loss'])), history2.history['loss'], label='Train P2')
axes[1].plot(range(offset, offset+len(history2.history['val_loss'])), history2.history['val_loss'], label='Val P2')
axes[1].set_title('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].hist(confidence_scores, bins=50, edgecolor='black', alpha=0.7)
axes[2].axvline(x=np.mean(confidence_scores), color='red', linestyle='--')
axes[2].set_title('Confianza')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'training_metrics.png'), dpi=150)
plt.show()

In [ ]:
# 12. Guardar modelo y labels
print("="*60)
print("💾 GUARDANDO MODELO")
print("="*60)

# Guardar Keras
model.save(str(OUTPUT_DIR / 'colombia_crop_disease.keras'))
print(f"✅ {OUTPUT_DIR / 'colombia_crop_disease.keras'}")

# Convertir a TFLite
print("\n🔄 Convirtiendo a TFLite...")

# Desactivar mixed precision para exportación
tf.keras.mixed_precision.set_global_policy('float32')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float32]

tflite_model = converter.convert()
tflite_path = OUTPUT_DIR / 'colombia_crop_disease.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

tflite_size = os.path.getsize(tflite_path) / (1024*1024)
print(f"✅ {tflite_path} ({tflite_size:.1f} MB)")

# Labels en español
class_names = list(train_gen.class_indices.keys())
labels = []

for i, name in enumerate(class_names):
    parts = name.split('___')
    crop = parts[0]
    disease = parts[1] if len(parts) > 1 else 'Saludable'
    
    labels.append({
        "id": i,
        "name": name,
        "crop": crop,
        "display": disease.replace('_', ' ')
    })

labels_path = OUTPUT_DIR / 'colombia_crop_labels.json'
with open(labels_path, 'w', encoding='utf-8') as f:
    json.dump({"labels": labels}, f, indent=2, ensure_ascii=False)

print(f"✅ {labels_path} ({len(labels)} clases)")

print("\n📋 Cultivos incluidos:")
for crop in sorted(set(l['crop'] for l in labels)):
    crop_diseases = [l['display'] for l in labels if l['crop'] == crop]
    print(f"   🌱 {crop}: {', '.join(crop_diseases)}")

In [ ]:
# 13. Resumen final
print("="*60)
print("🎉 ENTRENAMIENTO COMPLETADO EN AZURE")
print("="*60)

print(f"""
📊 RESULTADOS:
   • Accuracy: {final_acc:.2%}
   • Top-3 Accuracy: {final_top3:.2%}
   • Confianza media: {np.mean(confidence_scores):.2%}
   • Tiempo total: {total_time/60:.1f} minutos
   • Clases: {NUM_CLASSES}

🇨🇴 CULTIVOS COLOMBIANOS:
   ☕ Café (Roya, Cercospora, Minador)
   🥬 Yuca (Mosaico, Añublo, Ácaro)
   🥔 Papa (Tizón temprano/tardío)
   🌽 Maíz (Roya, Tizón norteño)
   🍅 Tomate (Virus, Tizón, Septoria)

📁 ARCHIVOS EN ./outputs/:
   • colombia_crop_disease.keras
   • colombia_crop_disease.tflite ({tflite_size:.1f} MB)
   • colombia_crop_labels.json
   • training_metrics.png
   • logs/ (TensorBoard)

═══════════════════════════════════════════════════════════
📥 DESCARGAR DESDE AZURE ML:
═══════════════════════════════════════════════════════════

Los archivos están en la carpeta ./outputs/ que se guarda
automáticamente en Azure ML.

Para descargar:
1. Ve a Azure ML Studio → Experiments → tu experimento
2. Click en el run completado
3. Tab "Outputs + logs"
4. Descarga los archivos .tflite y .json

O usa Azure CLI:
   az ml job download --name <job-id> --outputs

═══════════════════════════════════════════════════════════
📱 PARA ANDROID:
═══════════════════════════════════════════════════════════

1. Convierte a MindSpore:
   ./converter_lite --fmk=TFLITE \
       --modelFile=colombia_crop_disease.tflite \
       --outputFile=plant_disease_model

2. Copia a Android:
   cp plant_disease_model.ms app/src/main/assets/
   cp colombia_crop_labels.json app/src/main/assets/plant_disease_labels.json

═══════════════════════════════════════════════════════════
""")

# Listar archivos de salida
print("📂 Archivos generados:")
for f in OUTPUT_DIR.iterdir():
    if f.is_file():
        size = f.stat().st_size / (1024*1024)
        print(f"   {f.name}: {size:.2f} MB")

# 💰 Costos estimados en Azure ML

## Tiempo de entrenamiento por GPU:

| GPU | VM Size | Tiempo Est. | Precio/hora | **Costo Total** |
|-----|---------|-------------|-------------|------------------|
| K80 | NC6 | ~2.5 horas | $0.90 | ~$2.25 |
| V100 | NC6s_v3 | ~45 min | $3.06 | ~$2.30 |
| 2x V100 | NC12s_v3 | ~25 min | $6.12 | ~$2.55 |
| A100 | NC24ads_A100 | ~18 min | $3.67 | ~$1.10 |

## Recomendación:
**NC24ads_A100_v4** es la mejor opción:
- Más rápido (~18 min)
- Más barato (~$1.10)
- 80GB VRAM permite batch size grande
- Mixed precision automático

## Cómo crear el Compute en Azure ML:

```python
from azure.ai.ml import MLClient
from azure.ai.ml.entities import AmlCompute

# Crear compute A100
compute = AmlCompute(
    name="gpu-a100-cluster",
    size="Standard_NC24ads_A100_v4",
    min_instances=0,
    max_instances=1,
    idle_time_before_scale_down=300  # 5 min idle → apaga
)
ml_client.compute.begin_create_or_update(compute)
```

## TensorBoard en Azure:
Los logs se guardan en `./outputs/logs/`. Puedes verlos en:
- Azure ML Studio → Run → Metrics
- O localmente: `tensorboard --logdir ./outputs/logs`